# Week 9: AI Agents & ReAct

606363 Computing Systems for Data Science and AI — every code example from the lesson page, in order.

Lesson page: https://zmomani.github.io/ComputingSystemsDS-AI/lessons/week09-react-agents.html

**How to use:** run the cells from the top (Runtime → Run all, or Shift+Enter cell by cell). Later examples often reuse variables from earlier ones. Examples that call OpenAI or other APIs need a key: add it with the 🔑 Secrets panel on the left, or set `os.environ["OPENAI_API_KEY"] = "..."` in a cell. Examples that use Ollama need a local Ollama server and will not work in Colab. For GPU examples choose Runtime → Change runtime type → T4 GPU.

In [ ]:
# Libraries this lesson uses that are not preinstalled in Colab
%pip install -q openai python-dotenv

## 2. Setting Up the Environment

In [ ]:
# environment_setup.py
import openai
import re
import httpx
import os
from dotenv import load_dotenv

_ = load_dotenv()
from openai import OpenAI

client = OpenAI()

# Test the OpenAI client
chat_completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Hello world"}]
)

print(chat_completion.choices[0].message.content)

## 3. Creating the Basic Agent Class

In [ ]:
# agent.py
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
                        model="gpt-4o",
                        temperature=0,
                        messages=self.messages)
        return completion.choices[0].message.content

## 4. Defining the ReAct Prompt

In [ ]:
# react_prompt.py
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

## 5. Implementing the Tools

In [ ]:
# tools.py
import ast
import operator

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg, ast.UAdd: operator.pos}

def _safe_eval(node):
    # Walk the parsed expression tree; only numbers and arithmetic operators are allowed
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError(f"Unsupported expression: {ast.unparse(node)}")

def calculate(what):
    # Safe arithmetic evaluator (never call eval() on model output)
    return _safe_eval(ast.parse(what.strip(), mode="eval"))

def average_dog_weight(name):
    name = name.strip().lower()   # exact, case-insensitive match (not a substring test)
    if name == "scottish terrier":
        return("Scottish Terriers average 20 lbs")
    elif name == "border collie":
        return("a Border Collies average weight is 37 lbs")
    elif name == "toy poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

## 6. Creating the Agent Loop

In [ ]:
# agent_loop.py
action_re = re.compile(r'^Action: (\w+): (.*)$')   # python regular expression to select action

def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [
            action_re.match(a)
            for a in result.split('\n')
            if action_re.match(a)
        ]
        if actions:
            # There is an action to run
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return

## 7. Complete Example

In [ ]:
# complex_example.py
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""

query(question)
# The agent will:
# 1. Look up the weight of a border collie
# 2. Look up the weight of a scottish terrier
# 3. Calculate the sum of the weights
# 4. Provide the final answer

## 8. Complete Code Architecture

In [ ]:
# react_agent.py
import ast
import operator
import re
from dotenv import load_dotenv
from openai import OpenAI

_ = load_dotenv()
client = OpenAI()


class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
                        model="gpt-4o",
                        temperature=0,
                        messages=self.messages)
        return completion.choices[0].message.content


prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()


_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg, ast.UAdd: operator.pos}

def _safe_eval(node):
    # Walk the parsed expression tree; only numbers and arithmetic operators are allowed
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError(f"Unsupported expression: {ast.unparse(node)}")

def calculate(what):
    # Safe arithmetic evaluator (never call eval() on model output)
    return _safe_eval(ast.parse(what.strip(), mode="eval"))

def average_dog_weight(name):
    name = name.strip().lower()   # exact, case-insensitive match (not a substring test)
    if name == "scottish terrier":
        return("Scottish Terriers average 20 lbs")
    elif name == "border collie":
        return("a Border Collies average weight is 37 lbs")
    elif name == "toy poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

action_re = re.compile(r'^Action: (\w+): (.*)$')   # python regular expression to select action

def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [
            action_re.match(a)
            for a in result.split('\n')
            if action_re.match(a)
        ]
        if actions:
            # There is an action to run
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return


if __name__ == "__main__":
    question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
    query(question)

## Run in your browser

**A ReAct loop with a scripted model and real tools**

In [ ]:
import ast, operator, re

def calculate(expr):
    ops = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv, ast.Pow: operator.pow}
    def ev(n):
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in ops: return ops[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

def lookup(term):
    facts = {"amman population": "about 4.06 million (2024 estimate)"}
    return facts.get(term.lower(), "no result")

tools = {"calculate": calculate, "lookup": lookup}

script = iter([                       # what an LLM would generate, one turn at a time
    "Thought: I need the population first.\nAction: lookup: Amman population",
    "Thought: Now compute 17% of 4.06 million.\nAction: calculate: 0.17 * 4060000",
    "Thought: I have the answer.\nAnswer: about 690,200 people",
])
action_re = re.compile(r"^Action: (\w+): (.*)$")

for turn in range(1, 6):
    reply = next(script)
    print(reply)
    if reply.splitlines()[-1].startswith("Answer:"):
        break
    m = next(action_re.match(l) for l in reply.splitlines() if action_re.match(l))
    tool, arg = m.groups()
    print(f"Observation: {tools[tool](arg)}\n")

## Test code for the course environment

**Regex action extraction, no API key needed**

In [ ]:
# 1. Regex action extraction, no API key needed
import re
action_re = re.compile(r'^Action: (\w+): (.*)$')
sample = """Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE"""
actions = [action_re.match(a) for a in sample.split('\n') if action_re.match(a)]
print(actions[0].groups())            # ('average_dog_weight', 'Bulldog')
print(action_re.match("Answer: 57 lbs"))   # None → loop would terminate

**Tools + dispatch table, exercised directly**

In [ ]:
# 2. Tools + dispatch table, exercised directly
import ast, operator

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg, ast.UAdd: operator.pos}

def _safe_eval(node):
    # Walk the parsed expression tree; only numbers and arithmetic operators are allowed
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError(f"Unsupported expression: {ast.unparse(node)}")

def calculate(what):
    # Safe arithmetic evaluator (never call eval() on model output)
    return _safe_eval(ast.parse(what.strip(), mode="eval"))

def average_dog_weight(name):
    name = name.strip().lower()   # exact, case-insensitive match (not a substring test)
    if name == "scottish terrier":
        return("Scottish Terriers average 20 lbs")
    elif name == "border collie":
        return("a Border Collies average weight is 37 lbs")
    elif name == "toy poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = {"calculate": calculate, "average_dog_weight": average_dog_weight}
for action, arg in [("average_dog_weight", "Border Collie"),
                    ("average_dog_weight", "Scottish Terrier"),
                    ("calculate", "37 + 20"),
                    ("average_dog_weight", "scottish terrier"),   # lowercase still matches
                    ("average_dog_weight", "Terrier")]:           # partial name → default
    print(f"{action}({arg!r}) -> {known_actions[action](arg)}")

try:
    calculate("__import__('os').system('ls')")    # rejected by the safe evaluator
except ValueError as e:
    print("blocked:", e)

**Run the full ReAct loop offline with a scripted fake LLM (same Agent/query code, no OpenAI**

In [ ]:
# 3. Run the full ReAct loop offline with a scripted fake LLM (same Agent/query code, no OpenAI call)
#    (run test 2 first so that calculate() is defined)
import re
scripted = iter([
    "Thought: I need both weights.\nAction: average_dog_weight: Border Collie\nPAUSE",
    "Thought: Now the terrier.\nAction: average_dog_weight: Scottish Terrier\nPAUSE",
    "Thought: Add them.\nAction: calculate: 37 + 20\nPAUSE",
    "Answer: The combined weight is 57 lbs.",
])

class Agent:
    def __init__(self, system=""):
        self.messages = [{"role": "system", "content": system}] if system else []
    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = next(scripted)                     # stand-in for client.chat.completions.create
        self.messages.append({"role": "assistant", "content": result})
        return result

known_actions = {"calculate": calculate,                      # the safe ast-based calculate() from test 2
                 "average_dog_weight": lambda n: {"border collie": "a Border Collies average weight is 37 lbs",
                                                  "scottish terrier": "Scottish Terriers average 20 lbs"}.get(n.strip().lower(), "An average dog weights 50 lbs")}
action_re = re.compile(r'^Action: (\w+): (.*)$')

def query(question, max_turns=5):
    bot, next_prompt = Agent("ReAct prompt"), question
    for i in range(max_turns):
        result = bot(next_prompt); print(result)
        actions = [action_re.match(a) for a in result.split('\n') if action_re.match(a)]
        if not actions:
            return len(bot.messages)
        action, arg = actions[0].groups()
        obs = known_actions[action](arg); print(" -- running", action, arg); print("Observation:", obs)
        next_prompt = f"Observation: {obs}"

print("messages in history:", query("I have 2 dogs, a border collie and a scottish terrier. What is their combined weight"))

**Live run against OpenAI (needs OPENAI_API_KEY in .env): the lesson's own entry point**

In [ ]:
# 4. Live run against OpenAI (needs OPENAI_API_KEY in .env): the lesson's own entry point
#    pip install openai python-dotenv
from dotenv import load_dotenv; _ = load_dotenv()
from openai import OpenAI
client = OpenAI()
# paste the Agent class, prompt, tools, known_actions and query() from section 8 here, then:
# query("I have 2 dogs, a border collie and a scottish terrier. What is their combined weight")